# Bielik-4.5B — inference-only: prawdopodobieństwa val+test do ensemble

Bez treningu — ładuje wytrenowany adapter QLoRA i liczy macierze prawdopodobieństw do ensemble.

**Wymaga:** GPU T4 (bitsandbytes, CC≥7.5), Internet ON, datasety `pl-emotion-processed`

In [ ]:
!pip install -q -U "transformers>=4.44" "datasets>=2.20" accelerate peft bitsandbytes 2>/dev/null
import torch, transformers
print(transformers.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
# --- HF auth: sekret Kaggle (wymagany: Bielik-4.5B-v3.0 to repo gated) ---
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))
except Exception as e:
    HF_TOKEN = ""  # <-- jesli sekret nie podpiety, wklej token tu (nie commitowac!)
    assert HF_TOKEN, f"Brak sekretu HF_TOKEN ({e}) i brak tokenu inline"
    login(HF_TOKEN)
print("HF zalogowany")


In [ ]:
import glob, warnings
import numpy as np, pandas as pd, torch
from scipy.special import expit
from datasets import Dataset
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import PeftModel
warnings.filterwarnings("ignore")
EMOTIONS=["radość","smutek","zaufanie","wstręt","strach","gniew","przeczuwanie","zdziwienie"]
MODEL_NAME="speakleash/Bielik-4.5B-v3.0-Instruct"; MAX_LEN=128; OUT="/kaggle/working"

def find(name, pat):
    hits=glob.glob(f"/kaggle/input/**/{pat}", recursive=True)
    assert hits, f"{name}: {pat} nie znalezione w /kaggle/input"
    return hits[0]

tw_val =pd.read_csv(find("val","twitteremo_val.csv")).reset_index(drop=True)
tw_test=pd.read_csv(find("test","twitteremo_test.csv")).reset_index(drop=True)
for d in (tw_val,tw_test): d["tekst"]=d["tekst"].fillna("")
y_val,y_test=tw_val[EMOTIONS].values,tw_test[EMOTIONS].values
ADAPTER=find("adapter","adapter_config.json").rsplit("/",1)[0]
print("adapter:",ADAPTER)

In [ ]:
tok=AutoTokenizer.from_pretrained(ADAPTER)
if tok.pad_token is None: tok.pad_token=tok.eos_token
bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                       bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
base=AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(EMOTIONS), problem_type="multi_label_classification",
    quantization_config=bnb, device_map="auto")
base.config.pad_token_id=tok.pad_token_id
model=PeftModel.from_pretrained(base, ADAPTER).eval()
print("model zaladowany (4-bit + adapter, head w modules_to_save)")

In [ ]:
@torch.no_grad()
def predict(texts, bs=32):
    out=[]
    for i in range(0,len(texts),bs):
        enc=tok(list(texts[i:i+bs]),truncation=True,max_length=MAX_LEN,
                padding=True,return_tensors="pt").to(model.device)
        out.append(expit(model(**enc).logits.float().cpu().numpy()))
        if (i//bs)%20==0: print(f"  {i}/{len(texts)}",flush=True)
    return np.vstack(out)

p_val =predict(tw_val["tekst"].tolist())
p_test=predict(tw_test["tekst"].tolist())
np.save(f"{OUT}/bielik45_proba_val.npy",p_val)
np.save(f"{OUT}/bielik45_proba_test.npy",p_test)

# sanity: progi z val -> F1 test powinno wyjsc ~0.595 (jak w runie treningowym)
thr=np.full(len(EMOTIONS),0.5)
for i in range(len(EMOTIONS)):
    bf,bt=0.0,0.5
    for t in np.arange(0.05,0.95,0.01):
        f=f1_score(y_val[:,i],(p_val[:,i]>=t).astype(int),zero_division=0)
        if f>bf: bf,bt=f,t
    thr[i]=bt
f1=f1_score(y_test,(p_test>=thr).astype(int),average="macro",zero_division=0)
print(f"SANITY F1-Macro(test)={f1:.3f}  (oczekiwane ~0.595)")

## Wynik
`bielik45_proba_val.npy` + `bielik45_proba_test.npy` → skopiuj do `data/results/external_probas/`.